In [8]:
import wikipediaapi
import pandas as pd
from tqdm import tqdm
import time
import os
import re

os.makedirs("documents", exist_ok=True)

wiki = wikipediaapi.Wikipedia(
    user_agent='MyWikipediaRecommender/1.0 (https://github.com/MNOWAK1234)',
    language='en'
)

# Seed topics to start crawling from
seed_articles = [
    "Artificial intelligence", "Physics", "Mathematics", "Music",
    "Computer science", "Biology", "Chemistry", "Economics",
    "Psychology", "History"
]

articles_data = {}
max_articles = 1000

print("Collecting Wikipedia articles...")

def sanitize_filename(title: str) -> str:
    """Make a safe filename from article title."""
    filename = re.sub(r'[\\/*?:"<>|]', "_", title)  # replace forbidden characters
    return filename.strip().replace(" ", "_")

for seed in tqdm(seed_articles):
    page = wiki.page(seed)
    if not page.exists():
        continue

    # Get linked articles from this seed page
    for link_title in page.links.keys():
        if len(articles_data) >= max_articles:
            break

        # Skip if already saved
        if link_title in articles_data:
            continue

        linked_page = wiki.page(link_title)
        if linked_page.exists() and len(linked_page.text) > 500:
            title = linked_page.title
            url = linked_page.fullurl
            text = linked_page.text

            # Save to dictionary (for CSV)
            articles_data[title] = {
                "title": title,
                "url": url,
                "text": text
            }

            # Save to individual .txt file
            filename = sanitize_filename(title) + ".txt"
            filepath = os.path.join("documents", filename)
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(url + "\n\n")  # URL in first line, then blank line
                f.write(text)

            # Be polite: short delay every 50 requests
            if len(articles_data) % 50 == 0:
                time.sleep(1)

    if len(articles_data) >= max_articles:
        break

df = pd.DataFrame(articles_data.values())
df.to_csv("wikipedia_articles.csv", index=False, encoding="utf-8")

print(f"✅ Done! Saved {len(df)} articles to wikipedia_articles.csv and documents/ folder.")

  0%|          | 0/10 [07:47<?, ?it/s]


✅ Done! Saved 1000 articles to wikipedia_articles.csv and documents/ folder.
